# MNIST predictive coding — NumPy, in the planar PC style

**Architecture:** `784 → 128 → 64 → 10`. Examples are columns: `X (784, m)`, `Y (10, m)`.

Follow the same functions as `planar_pc_simple`: forward propagation, prediction errors, hidden-state inference, local parameter gradients, and weight updates. Hidden predictions still use **tanh**. The output now uses **softmax with categorical cross-entropy**.

Each mini-batch starts fresh activities from its forward pass. Inference adjusts activities with weights fixed; learning then adjusts weights with activities fixed. Prediction uses only a forward pass and never needs labels.

Only NumPy and Python's standard library are required. The first data load downloads MNIST (about 11 MB); later runs reuse a temporary-directory cache. Full training runs for 10 epochs and can take several minutes depending on your CPU. All settings are in the final training cell.

## 1. Import libraries and load MNIST

In [1]:
from pathlib import Path
from urllib.request import urlopen
import tempfile
import time
import numpy as np


def load_mnist(path=None):
    # Override path to use an existing archive or a permanent cache location.
    path = Path(path) if path is not None else Path(tempfile.gettempdir()) / "mnist_pc_simple" / "mnist.npz"
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz"
        print("Downloading MNIST to", path)
        with urlopen(url, timeout=60) as response:
            content = response.read()
        # Finish downloading before writing the cache.
        path.write_bytes(content)
    with np.load(path, allow_pickle=False) as data:
        x_train, y_train = data["x_train"], data["y_train"]
        x_test, y_test = data["x_test"], data["y_test"]
    def prepare(images, labels):
        X = images.reshape(-1, 784).T.astype(np.float32) / 255.0
        Y = np.eye(10, dtype=np.float32)[labels].T
        return X, Y
    return (*prepare(x_train, y_train), *prepare(x_test, y_test))


X_train, Y_train, X_test, Y_test = load_mnist()
print("Training:", X_train.shape, Y_train.shape)
print("Test:    ", X_test.shape, Y_test.shape)

Training: (784, 60000) (10, 60000)
Test:     (784, 10000) (10, 10000)


## 2. Define layers

The two hidden sizes change to 128 and 64. One-hot targets give `n_y = 10`.

In [2]:
def layers(X, Y):
    return X.shape[0], 128, 64, Y.shape[0]

## 3. Initialize weights and biases

Use zero-centered Xavier normal weights, with standard deviation `sqrt(2 / (fan_in + fan_out))`, for all three layers. This gives the larger tanh network a useful starting scale. Biases start at zero. A local random generator makes initialization reproducible without changing NumPy's global seed.

In [3]:
def initialize(n_x, h1, h2, n_y, seed=3):
    rng = np.random.default_rng(seed)
    parameters = {}
    for i, (fan_in, fan_out) in enumerate(zip((n_x, h1, h2), (h1, h2, n_y)), 1):
        parameters["W" + str(i)] = (rng.standard_normal((fan_out, fan_in)) * np.sqrt(2.0 / (fan_in + fan_out))).astype(np.float32)
        parameters["b" + str(i)] = np.zeros((fan_out, 1), dtype=np.float32)
    return parameters

## 4. Forward propagation

Keep `forwared_propagation` spelled as in the planar notebook for easy comparison. Softmax normalizes **over classes in each column**. Subtracting the largest logit prevents exponential overflow.

$$p_{k,j}=\frac{e^{Z_{k,j}-\max_r Z_{r,j}}}{\sum_r e^{Z_{r,j}-\max_r Z_{r,j}}}.$$

Matrix products use `np.dot`, equivalent to `@` for these 2D arrays. This also avoids spurious `matmul` floating-point warnings in the local NumPy 2.0 / Apple Accelerate setup; no warnings are suppressed.

In [4]:
def softmax(Z):
    shifted = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(shifted)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)


def forwared_propagation(X, parameters):
    Z1 = np.dot(parameters["W1"], X) + parameters["b1"]
    A1 = np.tanh(Z1)
    Z2 = np.dot(parameters["W2"], A1) + parameters["b2"]
    A2 = np.tanh(Z2)
    Z3 = np.dot(parameters["W3"], A2) + parameters["b3"]
    A3 = softmax(Z3)
    return {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2, "Z3": Z3, "A3": A3}

## 5a. Calculate prediction errors

Hidden errors remain **activity minus prediction**. For softmax with one-hot cross-entropy, the derivative with respect to output logits is still $dZ_3=p-Y$. Do not multiply it by a separate softmax derivative.

$$\mu_1=\tanh(W_1X+b_1),\quad e_1=H_1-\mu_1;\qquad
\mu_2=\tanh(W_2H_1+b_2),\quad e_2=H_2-\mu_2.$$

The output prediction uses current inferred `H2`, and `mu2` uses current inferred `H1`.

In [5]:
def compute_errors(X, Y, parameters, H1, H2):
    mu1 = np.tanh(np.dot(parameters["W1"], X) + parameters["b1"])
    mu2 = np.tanh(np.dot(parameters["W2"], H1) + parameters["b2"])
    Z3 = np.dot(parameters["W3"], H2) + parameters["b3"]
    p = softmax(Z3)
    return {"mu1": mu1, "mu2": mu2, "Z3": Z3, "p": p,
            "e1": H1 - mu1, "e2": H2 - mu2, "dZ3": p - Y}

## 5b. Calculate classification cost and PC energy

Compute cross-entropy from shifted logits, avoiding `log(0)` even when softmax probabilities underflow. Sum over classes, then average over examples.

$$\mathrm{CE}(Y,Z)=\frac1m\sum_j\left[\log\sum_k e^{S_{k,j}}-\sum_kY_{k,j}S_{k,j}\right],\quad S_{k,j}=Z_{k,j}-\max_rZ_{r,j}.$$

$$E=\frac{\lVert e_1\rVert_F^2+\lVert e_2\rVert_F^2}{2m}+\mathrm{CE}(Y,Z_3).$$

Forward cross-entropy measures classification. PC energy measures the inference objective; these are different after hidden activities move.

In [6]:
def compute_cost(Y, cache):
    shifted = cache["Z3"] - np.max(cache["Z3"], axis=0, keepdims=True)
    log_normalizer = np.log(np.sum(np.exp(shifted), axis=0))
    return np.mean(log_normalizer - np.sum(Y * shifted, axis=0))


def compute_energy(Y, errors):
    hidden_loss = 0.5 * (np.sum(errors["e1"]**2) + np.sum(errors["e2"]**2))
    return hidden_loss / Y.shape[1] + compute_cost(Y, errors)

## 5c. Infer hidden states

The planar equations carry over directly:

$$g_{H_2}=e_2+W_3^T(p-Y),\qquad
g_{H_1}=e_1-W_2^T\left[e_2\odot(1-\mu_2^2)\right].$$

These are **per-example** energy derivatives, without division by batch size. Calculate both from the old state before updating either activity. Weights remain fixed. Activities are unconstrained variables; do not apply an extra tanh or clip them. Recompute errors at the final state before weight learning.

In [7]:
def inference(X, Y, parameters, cache, steps=20, activity_rate=0.05):
    H1, H2 = cache["A1"].copy(), cache["A2"].copy()
    for _ in range(steps):
        errors = compute_errors(X, Y, parameters, H1, H2)
        dH2 = errors["e2"] + np.dot(parameters["W3"].T, errors["dZ3"])
        dH1 = errors["e1"] - np.dot(parameters["W2"].T, errors["e2"] * (1 - errors["mu2"]**2))
        H1 = H1 - activity_rate * dH1
        H2 = H2 - activity_rate * dH2
    return H1, H2, compute_errors(X, Y, parameters, H1, H2)

## 6a. Calculate local weight gradients

Hold inferred activities fixed. The equations and sign convention match planar PC:

$$dZ_1=-e_1\odot(1-\mu_1^2),\quad dZ_2=-e_2\odot(1-\mu_2^2),\quad dZ_3=p-Y.$$

$$dW_1=dZ_1X^T/m,\quad dW_2=dZ_2H_1^T/m,\quad dW_3=dZ_3H_2^T/m.$$

Divide weight and bias gradients by the **actual** batch size, including the smaller last batch. No differentiation through the inference loop is needed.

In [8]:
def parameter_gradients(X, Y, parameters, H1, H2, errors):
    m = X.shape[1]
    dZ1 = -errors["e1"] * (1 - errors["mu1"]**2)
    dZ2 = -errors["e2"] * (1 - errors["mu2"]**2)
    dZ3 = errors["dZ3"]
    return {"dW1": (np.dot(dZ1, X.T)) / m, "db1": np.sum(dZ1, axis=1, keepdims=True) / m,
            "dW2": (np.dot(dZ2, H1.T)) / m, "db2": np.sum(dZ2, axis=1, keepdims=True) / m,
            "dW3": (np.dot(dZ3, H2.T)) / m, "db3": np.sum(dZ3, axis=1, keepdims=True) / m}

## 6b. Update weights — the same subtraction

In [9]:
def weight_update(parameters, grads, weight_rate=0.1):
    for name in ("W1", "b1", "W2", "b2", "W3", "b3"):
        parameters[name] = parameters[name] - weight_rate * grads["d" + name]
    return parameters

## 7. Mini-batches, prediction, and evaluation

Shuffle matching columns of `X` and `Y` together once per epoch. Yield the final partial batch too. Prediction returns digit IDs `(m,)` using `argmax` instead of a binary threshold.

Evaluation uses forward passes in batches, accumulating sample-weighted loss and accuracy. This avoids allocating full-dataset hidden states. Test labels are used only to score predictions.

In [10]:
def mini_batches(X, Y, batch_size, rng):
    if batch_size <= 0 or X.shape[1] == 0 or X.shape[1] != Y.shape[1]:
        raise ValueError("Use a positive batch size and nonempty matching X/Y columns.")
    order = rng.permutation(X.shape[1])
    for start in range(0, X.shape[1], batch_size):
        indices = order[start:start + batch_size]
        yield X[:, indices], Y[:, indices]


def predict(X, parameters):
    return np.argmax(forwared_propagation(X, parameters)["A3"], axis=0)


def evaluate(X, Y, parameters, batch_size=512):
    if batch_size <= 0 or X.shape[1] == 0 or X.shape[1] != Y.shape[1]:
        raise ValueError("Use a positive batch size and nonempty matching X/Y columns.")
    loss, correct = 0.0, 0
    for start in range(0, X.shape[1], batch_size):
        xb, yb = X[:, start:start + batch_size], Y[:, start:start + batch_size]
        cache = forwared_propagation(xb, parameters)
        loss += float(compute_cost(yb, cache)) * xb.shape[1]
        correct += int(np.sum(np.argmax(cache["A3"], axis=0) == np.argmax(yb, axis=0)))
    return loss / X.shape[1], correct / X.shape[1]

## 8. Integrate training

Replace the planar iteration loop with epochs over mini-batches. For **each batch**: forward pass → fresh activities → inference → local gradients → one weight update. Reuse the weights across batches.

Record `(epoch, forward_cross_entropy, forward_accuracy)` before training and after every epoch. Test data never enters training. A finite-value check stops an unstable run with a useful message; it does not silently clip gradients or change the algorithm.

In [11]:
def pc_model(X, Y, h1=128, h2=64, epochs=10, batch_size=128,
             steps=20, activity_rate=0.05, weight_rate=0.1, seed=3):
    if (epochs < 0 or steps < 1 or batch_size < 1 or activity_rate <= 0 or weight_rate <= 0
            or X.shape[1] == 0 or X.shape[1] != Y.shape[1]):
        raise ValueError("Use matching nonempty X/Y, nonnegative epochs, and positive steps, batch size, and rates.")
    n_x, _, _, n_y = layers(X, Y)
    parameters = initialize(n_x, h1, h2, n_y, seed)
    rng = np.random.default_rng(seed)
    history = []
    start_time = time.perf_counter()
    for epoch in range(epochs + 1):
        if epoch > 0:
            for xb, yb in mini_batches(X, Y, batch_size, rng):
                cache = forwared_propagation(xb, parameters)
                H1, H2, errors = inference(xb, yb, parameters, cache, steps, activity_rate)
                grads = parameter_gradients(xb, yb, parameters, H1, H2, errors)
                parameters = weight_update(parameters, grads, weight_rate)
                if not all(np.all(np.isfinite(v)) for v in parameters.values()):
                    raise FloatingPointError("Nonfinite parameters; reduce activity_rate or weight_rate.")
        loss, accuracy = evaluate(X, Y, parameters)
        if not np.isfinite(loss):
            raise FloatingPointError("Nonfinite forward loss; reduce activity_rate or weight_rate.")
        history.append((epoch, loss, accuracy))
        print(f"Epoch {epoch:2d}: CE = {loss:.5f}, accuracy = {accuracy:.2%}, elapsed = {time.perf_counter() - start_time:.1f}s", flush=True)
    return parameters, history

## 9. Tiny numerical checks before training

Use a small float64 problem with perturbed hidden activities. Central finite differences check every activity and parameter entry. For parameter checks the activities stay fixed. Activity derivatives divide by `m` only here, because we differentiate the **mean** energy.

Also check extreme logits, softmax normalization, mini-batch alignment and coverage, label-free prediction, and that inference preserves weights and lowers energy for a small step.

In [12]:
def sanity_checks():
    rng = np.random.default_rng(7)
    X = rng.normal(size=(4, 3))
    Y = np.eye(3)[:, [0, 2, 1]]
    params = {k: v.astype(np.float64) for k, v in initialize(4, 3, 2, 3).items()}
    cache = forwared_propagation(X, params)
    H1 = cache["A1"] + rng.normal(scale=0.1, size=(3, 3))
    H2 = cache["A2"] + rng.normal(scale=0.1, size=(2, 3))
    errors = compute_errors(X, Y, params, H1, H2)
    dH1 = errors["e1"] - np.dot(params["W2"].T, errors["e2"] * (1 - errors["mu2"]**2))
    dH2 = errors["e2"] + np.dot(params["W3"].T, errors["dZ3"])
    grads = parameter_gradients(X, Y, params, H1, H2, errors)
    def energy():
        return compute_energy(Y, compute_errors(X, Y, params, H1, H2))
    def check(array, expected):
        numerical = np.zeros_like(array)
        eps = 1e-6
        for index in np.ndindex(array.shape):
            original = array[index]
            array[index] = original + eps
            plus = energy()
            array[index] = original - eps
            minus = energy()
            array[index] = original
            numerical[index] = (plus - minus) / (2 * eps)
        np.testing.assert_allclose(numerical, expected, rtol=1e-4, atol=1e-7)
    check(H1, dH1 / X.shape[1])
    check(H2, dH2 / X.shape[1])
    for name in params:
        check(params[name], grads["d" + name])
    saved = {k: v.copy() for k, v in params.items()}
    initial_errors = compute_errors(X, Y, params, cache["A1"], cache["A2"])
    np.testing.assert_allclose(initial_errors["e1"], 0)
    np.testing.assert_allclose(initial_errors["e2"], 0)
    _, _, final_errors = inference(X, Y, params, cache, steps=5, activity_rate=0.01)
    assert compute_energy(Y, final_errors) < compute_energy(Y, initial_errors)
    for name in params:
        np.testing.assert_array_equal(params[name], saved[name])
    extreme = np.array([[10000., -10000.], [-10000., 10000.], [0., 0.]])
    assert np.all(np.isfinite(softmax(extreme)))
    np.testing.assert_allclose(softmax(extreme).sum(axis=0), 1)
    assert np.isfinite(compute_cost(np.eye(3)[:, [1, 0]], {"Z3": extreme}))
    marker = np.arange(11)[None, :]
    batches = list(mini_batches(marker, marker + 100, 4, np.random.default_rng(3)))
    assert [xb.shape[1] for xb, _ in batches] == [4, 4, 3]
    np.testing.assert_array_equal(np.sort(np.concatenate([xb.ravel() for xb, _ in batches])), marker.ravel())
    for xb, yb in batches:
        np.testing.assert_array_equal(yb, xb + 100)
    first = list(mini_batches(marker, marker, 4, np.random.default_rng(3)))
    np.testing.assert_array_equal(batches[0][0], first[0][0])
    loss, accuracy = evaluate(X, Y, params, batch_size=2)
    np.testing.assert_allclose(loss, compute_cost(Y, cache))
    np.testing.assert_allclose(accuracy, np.mean(predict(X, params) == np.argmax(Y, axis=0)))
    assert predict(X, params).shape == (3,)
    print("Passed: gradients, inference energy, fixed weights, stable softmax/CE, batching, and evaluation.")


sanity_checks()

Passed: gradients, inference energy, fixed weights, stable softmax/CE, batching, and evaluation.


## 10. Train on full MNIST

One epoch visits every training example once. Defaults use all 60,000 training images, mini-batches of 128, and 20 activity updates per weight update. Both learning rates are explicit because they control different loops.

The final test score uses the separate 10,000-image test set with ordinary forward prediction. Accuracy depends on training settings; no equality to backpropagation is assumed.

In [13]:
parameters, history = pc_model(
    X_train, Y_train, h1=128, h2=64, epochs=10, batch_size=128,
    steps=20, activity_rate=0.05, weight_rate=0.1, seed=3,
)
test_loss, test_accuracy = evaluate(X_test, Y_test, parameters)
print(f"Test: CE = {test_loss:.5f}, accuracy = {test_accuracy:.2%}")

Epoch  0: CE = 2.46954, accuracy = 6.90%, elapsed = 0.0s
Epoch  1: CE = 0.38488, accuracy = 88.63%, elapsed = 1.2s
Epoch  2: CE = 0.32668, accuracy = 90.15%, elapsed = 2.3s
Epoch  3: CE = 0.30062, accuracy = 91.03%, elapsed = 3.3s
Epoch  4: CE = 0.28384, accuracy = 91.52%, elapsed = 4.4s
Epoch  5: CE = 0.26926, accuracy = 92.02%, elapsed = 5.5s
Epoch  6: CE = 0.26024, accuracy = 92.29%, elapsed = 6.6s
Epoch  7: CE = 0.25085, accuracy = 92.62%, elapsed = 7.6s
Epoch  8: CE = 0.23960, accuracy = 92.95%, elapsed = 8.7s
Epoch  9: CE = 0.23111, accuracy = 93.20%, elapsed = 9.8s
Epoch 10: CE = 0.22291, accuracy = 93.45%, elapsed = 10.9s
Test: CE = 0.22853, accuracy = 93.39%


## Inspect a few predictions

Each returned value is a digit from 0 to 9. Labels are shown alongside only for comparison.

In [14]:
print("Predicted:", predict(X_test[:, :20], parameters))
print("Actual:   ", np.argmax(Y_test[:, :20], axis=0))

Predicted: [7 2 1 0 4 1 4 9 6 9 0 6 9 0 1 5 9 7 3 4]
Actual:    [7 2 1 0 4 1 4 9 5 9 0 6 9 0 1 5 9 7 3 4]
